### Observação sobre importações

As seguintes importações foram identificadas como não utilizadas no notebook:
- `classification_report`
- `LogisticRegression`
- `RandomizedSearchCV`
- `pickle`
- `confusion_matrix`
- `seaborn as sns`
- `matplotlib.pyplot as plt`
- `shap`

As importações foram mantidas para preservar o código original.


### Carregamento dos dados
Lê os conjuntos de treino e teste previamente preparados.

In [1]:
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import shap
import joblib
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Separação das variáveis
Identifica automaticamente as colunas WOE e as demais colunas para treinamento dos modelos.

In [2]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")

X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

In [3]:
cols_woes = []
cols_norm = []
for col in X_train.columns:
    if "_woe" in col:
        cols_woes.append(col)
    else:
        cols_norm.append(col)

## Model LGBM

Treinamento e comparação dos modelos LightGBM.

### Bibliotecas para otimização
Importa os recursos necessários para busca de hiperparâmetros utilizando Optuna.

In [4]:
skf = StratifiedKFold(    n_splits=6,    shuffle=True,   random_state=42 )
lgb_model_norm = lgb.LGBMClassifier(objective="binary",metric="auc")
lgb_model_woe = lgb.LGBMClassifier(objective="binary",metric="auc")

### Função de otimização
Define a função responsável por encontrar os melhores hiperparâmetros do LightGBM utilizando validação cruzada.

In [5]:
import optuna
import lightgbm as lgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score
)

### Treinamento do modelo com variáveis normais
Executa a otimização e treina o modelo utilizando as variáveis originais.

In [6]:
def optimize_lgbm(X, y, n_trials=30):

    def objective(trial):

        params = {
            "objective": "binary",
            "metric": "auc",
            "random_state": 42,
            "verbosity": -1,

            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                1000
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

            "num_leaves": trial.suggest_int(
                "num_leaves",
                15,
                63
            ),

            "min_child_samples": trial.suggest_int(
                "min_child_samples",
                20,
                100
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            )
        }

        model = lgb.LGBMClassifier(**params)

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        auc = cross_val_score(
            model,
            X,
            y,
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1
        ).mean()

        return auc

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    return study.best_params

### Treinamento do modelo com variáveis WOE
Executa a otimização e treina o modelo utilizando as variáveis transformadas por WOE.

In [7]:
best_params_norm = optimize_lgbm(
    X_train[cols_norm],
    y_train,
    n_trials=30
)

lgb_norm = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    random_state=42,
    verbosity=-1,
    **best_params_norm
)

lgb_norm.fit(
    X_train[cols_norm],
    y_train
)

[I 2026-06-09 22:57:51,069] A new study created in memory with name: no-name-21f4a56b-1740-4d29-a75a-273810e8fbfd
Best trial: 0. Best value: 0.860834:   3%|▎         | 1/30 [00:10<05:04, 10.50s/it]

[I 2026-06-09 22:58:01,563] Trial 0 finished with value: 0.8608336871587184 and parameters: {'n_estimators': 646, 'learning_rate': 0.030782695068089675, 'num_leaves': 40, 'min_child_samples': 90, 'subsample': 0.9281313001116032, 'colsample_bytree': 0.751994493356469}. Best is trial 0 with value: 0.8608336871587184.


Best trial: 1. Best value: 0.861509:   7%|▋         | 2/30 [00:20<04:37,  9.93s/it]

[I 2026-06-09 22:58:11,091] Trial 1 finished with value: 0.8615087579363788 and parameters: {'n_estimators': 647, 'learning_rate': 0.013850074337548418, 'num_leaves': 57, 'min_child_samples': 73, 'subsample': 0.7043791327341149, 'colsample_bytree': 0.9099314636907517}. Best is trial 1 with value: 0.8615087579363788.


Best trial: 1. Best value: 0.861509:  10%|█         | 3/30 [00:25<03:35,  7.99s/it]

...

Best trial: 22. Best value: 0.86504:  97%|█████████▋| 29/30 [01:52<00:03,  3.50s/it]

[I 2026-06-09 22:59:43,662] Trial 28 finished with value: 0.8649869265597516 and parameters: {'n_estimators': 550, 'learning_rate': 0.019845221961032222, 'num_leaves': 15, 'min_child_samples': 43, 'subsample': 0.7980667194763567, 'colsample_bytree': 0.7238077111455139}. Best is trial 22 with value: 0.8650402040333253.


Best trial: 22. Best value: 0.86504: 100%|██████████| 30/30 [02:01<00:00,  4.05s/it]
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[I 2026-06-09 22:59:52,673] Trial 29 finished with value: 0.8646187790065177 and parameters: {'n_estimators': 884, 'learning_rate': 0.018728323865669506, 'num_leaves': 15, 'min_child_samples': 28, 'subsample': 0.8281859480339543, 'colsample_bytree': 0.7227129653995648}. Best is trial 22 with value: 0.8650402040333253.


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,-1
,learning_rate,0.017675076740165812
,n_estimators,331
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,91


### Função de avaliação
Calcula métricas de desempenho como AUC, Gini e Accuracy.

In [8]:
best_params_woe = optimize_lgbm(
    X_train[cols_woes],
    y_train,
    n_trials=30
)

lgb_woe = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    random_state=42,
    verbosity=-1,
    **best_params_woe
)

lgb_woe.fit(
    X_train[cols_woes],
    y_train
)

[I 2026-06-09 22:59:54,846] A new study created in memory with name: no-name-fb72d6f5-dfbf-4a51-a8ba-c2354ef69c68
Best trial: 0. Best value: 0.860744:   3%|▎         | 1/30 [00:01<00:42,  1.46s/it]

[I 2026-06-09 22:59:56,309] Trial 0 finished with value: 0.8607441875426793 and parameters: {'n_estimators': 135, 'learning_rate': 0.010164906106060677, 'num_leaves': 34, 'min_child_samples': 94, 'subsample': 0.8550984943099098, 'colsample_bytree': 0.8284021476672495}. Best is trial 0 with value: 0.8607441875426793.


Best trial: 1. Best value: 0.861065:   7%|▋         | 2/30 [00:02<00:40,  1.46s/it]

[I 2026-06-09 22:59:57,770] Trial 1 finished with value: 0.8610653832674927 and parameters: {'n_estimators': 162, 'learning_rate': 0.05160954167059618, 'num_leaves': 45, 'min_child_samples': 67, 'subsample': 0.9305539012145427, 'colsample_bytree': 0.996103072738501}. Best is trial 1 with value: 0.8610653832674927.


Best trial: 1. Best value: 0.861065:  10%|█         | 3/30 [00:09<01:39,  3.68s/it]

...

Best trial: 19. Best value: 0.863161:  97%|█████████▋| 29/30 [01:26<00:01,  1.40s/it]

[I 2026-06-09 23:01:21,162] Trial 28 finished with value: 0.8614028044540177 and parameters: {'n_estimators': 100, 'learning_rate': 0.016565535276172598, 'num_leaves': 24, 'min_child_samples': 36, 'subsample': 0.7675873518479648, 'colsample_bytree': 0.7059367748661366}. Best is trial 19 with value: 0.8631613054398111.


Best trial: 19. Best value: 0.863161: 100%|██████████| 30/30 [01:28<00:00,  2.96s/it]
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[I 2026-06-09 23:01:23,680] Trial 29 finished with value: 0.8623088750716548 and parameters: {'n_estimators': 282, 'learning_rate': 0.0382070793605442, 'num_leaves': 31, 'min_child_samples': 32, 'subsample': 0.856889367295502, 'colsample_bytree': 0.7858899927840566}. Best is trial 19 with value: 0.8631613054398111.


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,-1
,learning_rate,0.03199133137854626
,n_estimators,218
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,21


### Comparação dos modelos
Avalia os dois modelos treinados e organiza os resultados em um DataFrame.

In [9]:
def evaluate_model(
    model,
    X_test,
    y_test,
    model_name
):

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = model.predict(X_test)

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    gini = 2 * auc - 1

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    return {
        "model": model_name,
        "auc": round(auc, 4),
        "gini": round(gini, 4),
        "accuracy": round(accuracy, 4)
    }

### Exibição dos resultados
Mostra a tabela consolidada com as métricas dos modelos.

In [10]:
result_norm = evaluate_model(
    lgb_norm,
    X_test[cols_norm],
    y_test,
    "LGBM_NORM"
)

result_woe = evaluate_model(
    lgb_woe,
    X_test[cols_woes],
    y_test,
    "LGBM_WOE"
)

results = pd.DataFrame([
    result_norm,
    result_woe
])




In [11]:
joblib.dump(lgb_norm, "../assets/models/lgb_norm.joblib")
joblib.dump(lgb_woe, "../assets/models/lgb_woe.joblib")

['../assets/models/lgb_woe.joblib']

### Célula reservada
Célula vazia disponível para futuras análises.

In [12]:
results

,model,auc,gini,accuracy
0,LGBM_NORM,0.8665,0.7331,0.9374
1,LGBM_WOE,0.8633,0.7267,0.9363
